# Kairos Dataset — Engagement Prediction

**SOMIN Lab 1**

Predict whether a LinkedIn post performs below (0), near (1), or above (2) its author's usual engagement. The main metric is macro-F1; QWK is secondary.

The solution combines CatBoost features with word and character TF-IDF classifiers. Models and ensemble weights are selected on an author-disjoint validation split. Feature engineering follows Tutorial 1–2 (Q1); prediction fusion follows Q7.


## 1. Load data


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score, classification_report, confusion_matrix
from kairos import features
from kairos.data import load_posts, load_followers
from kairos.label import author_level

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
labels = pd.read_csv(DATA_DIR / "labels.csv")
levelled = author_level(load_posts(DATA_DIR))
post_features = features.build(levelled, load_followers(DATA_DIR))
table = labels.merge(post_features, on=["author", "post"], how="left", validate="one_to_one")
print("Dataset:", table.shape)
print("Split sizes:", table["split"].value_counts().to_dict())

posts:  12%|█▏        | 664/5433 [00:01<00:10, 473.50file/s]


KeyboardInterrupt: 

In [ ]:
assert labels[["author", "post"]].notna().all().all()
assert not labels.duplicated(["author", "post"]).any()
assert table["level"].notna().all(), "Missing author history: check the dataset"
assert set(table["split"].unique()) == {"train", "test"}
assert set(table.loc[table["split"] == "train", "author"]).isdisjoint(
    set(table.loc[table["split"] == "test", "author"])
)

ordered = levelled.sort_values(["author", "posted_at", "post"], kind="stable")
assert levelled[["author", "post"]].reset_index(drop=True).equals(
    ordered[["author", "post"]].reset_index(drop=True)
), "author_level must return posts in chronological order"
print("Data integrity, author separation, and history order: OK")


## 2. Author-disjoint validation split


In [ ]:
train_authors = table.loc[table["split"] == "train", "author"].unique()
train_ids, val_ids = train_test_split(train_authors, test_size=0.2, random_state=42)
train_mask = (table["split"] == "train") & table["author"].isin(train_ids)
val_mask = (table["split"] == "train") & table["author"].isin(val_ids)
full_train_mask = table["split"] == "train"
test_mask = table["split"] == "test"
y_train = table.loc[train_mask, "label"]
y_val = table.loc[val_mask, "label"]
y_test = table.loc[test_mask, "label"]
print("Train:", train_mask.sum(), "Validation:", val_mask.sum(), "Test:", test_mask.sum())
print("Overlapping authors:", len(set(train_ids) & set(val_ids)))

In [ ]:
assert train_mask.sum() + val_mask.sum() == full_train_mask.sum()
assert not (train_mask & val_mask).any()
assert set(table.loc[train_mask, "author"]).isdisjoint(set(table.loc[val_mask, "author"]))
assert not (full_train_mask & test_mask).any()
print("Validation split integrity: OK")


## 3. Feature engineering

Historical features exclude the current post and use earlier posts from the same author. Reactions come from the dataset snapshot, so this is not real-time forecasting.


In [ ]:
NUMERIC = features.CONTENT_NUM + features.HISTORY_NUM + features.ARTEFACT_NUM
CATEGORICAL = features.CONTENT_CAT
text = table["text"].fillna("")
table["n_exclamations"] = text.str.count("!")
table["n_emojis"] = text.str.count(r"[\U0001F300-\U0001FAFF]")
table["n_uppercase"] = text.str.count(r"[A-Z]")
table["n_ellipsis"] = text.str.count(r"\.\.\.")
words = text.str.findall(r"\b\w+\b")
table["avg_word_length"] = words.apply(lambda w: sum(len(x) for x in w) / len(w) if w else 0)
table["unique_word_ratio"] = words.apply(lambda w: len(set(x.lower() for x in w)) / len(w) if w else 0)
table["uppercase_ratio"] = table["n_uppercase"] / table["n_chars"].clip(lower=1)
table["n_sentences"] = text.str.count(r"[.!?]+")
table["n_digits"] = text.str.count(r"\d")
EXTRA_TEXT_FEATURES = ["n_exclamations", "n_emojis", "n_uppercase", "n_ellipsis", "avg_word_length", "unique_word_ratio", "uppercase_ratio", "n_sentences", "n_digits"]
print("Text features:", len(EXTRA_TEXT_FEATURES))

In [ ]:
import numpy as np
import pandas as pd

activity = levelled[["author", "post", "posted_at"]].copy()
activity = activity.sort_values(["author", "posted_at", "post"])

def calculate_activity(group):
    dates = group["posted_at"]

    last_7 = []
    last_30 = []
    mean_gaps = []

    for i in range(len(group)):
        previous = dates.iloc[:i]

        last_7.append(
            ((previous >= dates.iloc[i] - pd.Timedelta(days=7)) &
             (previous < dates.iloc[i])).sum()
        )

        last_30.append(
            ((previous >= dates.iloc[i] - pd.Timedelta(days=30)) &
             (previous < dates.iloc[i])).sum()
        )

        recent = previous.tail(5)
        gaps = recent.diff().dt.total_seconds() / 86400

        mean_gaps.append(gaps.mean() if len(recent) >= 2 else -1.0)

    group = group.copy()
    group["posts_last_7d"] = last_7
    group["posts_last_30d"] = last_30
    group["mean_gap_last_5"] = mean_gaps

    return group

activity = pd.concat(
    [calculate_activity(group) for _, group in activity.groupby("author")],
    ignore_index=True
)

ACTIVITY_FEATURES = [
    "posts_last_7d",
    "posts_last_30d",
    "mean_gap_last_5"
]

table = table.merge(
    activity[["author", "post"] + ACTIVITY_FEATURES],
    on=["author", "post"],
    how="left",
    validate="one_to_one"
)

### Historical engagement


In [ ]:
history = levelled[
    ["author", "post", "log_reactions"]
].copy()

history["recent_mean_3"] = history.groupby("author")["log_reactions"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)

history["recent_mean_5"] = history.groupby("author")["log_reactions"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

history["recent_std_5"] = history.groupby("author")["log_reactions"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=2).std()
)

history["last_reaction"] = history.groupby("author")["log_reactions"].shift(1)

history["recent_trend"] = (
    history["recent_mean_3"] - history["recent_mean_5"]
)

REACTION_FEATURES = [
    "recent_mean_3",
    "recent_mean_5",
    "recent_std_5",
    "last_reaction",
    "recent_trend"
]

table = table.merge(
    history[["author", "post"] + REACTION_FEATURES],
    on=["author", "post"],
    how="left",
    validate="one_to_one"
)

### Content-type history


In [ ]:
content_history = levelled[
    ["author", "post", "content_type"]
].copy()

groups = content_history.groupby("author")

content_history["previous_same_type"] = (
    content_history.groupby(["author", "content_type"]).cumcount()
)

content_history["same_type_ratio"] = (
    content_history["previous_same_type"]
    / groups.cumcount().replace(0, np.nan)
)

content_history["same_as_previous"] = (
    content_history["content_type"]
    == groups["content_type"].shift(1)
).astype(int)

CONTENT_HISTORY_FEATURES = [
    "previous_same_type",
    "same_type_ratio",
    "same_as_previous"
]

table = table.merge(
    content_history[["author", "post"] + CONTENT_HISTORY_FEATURES],
    on=["author", "post"],
    how="left",
    validate="one_to_one"
)

In [ ]:
type_history = levelled[
    ["author", "post", "content_type", "log_reactions", "level"]
].copy()

type_groups = type_history.groupby(["author", "content_type"])["log_reactions"]

type_history["type_reaction_mean"] = type_groups.transform(
    lambda s: s.shift(1).expanding(min_periods=1).mean()
)

type_history["type_reaction_count"] = (
    type_history.groupby(["author", "content_type"]).cumcount()
)

type_history["type_vs_author"] = (
    type_history["type_reaction_mean"] - type_history["level"]
)

TYPE_PERFORMANCE_FEATURES = [
    "type_reaction_mean",
    "type_reaction_count",
    "type_vs_author"
]

table = table.merge(
    type_history[["author", "post"] + TYPE_PERFORMANCE_FEATURES],
    on=["author", "post"],
    how="left",
    validate="one_to_one"
)

In [ ]:
recent_type = levelled[
    ["author", "post", "content_type", "log_reactions", "level"]
].copy()

groups = recent_type.groupby(
    ["author", "content_type"]
)["log_reactions"]

recent_type["recent_type_mean_3"] = groups.transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)

recent_type["recent_type_std_3"] = groups.transform(
    lambda s: s.shift(1).rolling(3, min_periods=2).std()
)

recent_type["recent_type_vs_author"] = (
    recent_type["recent_type_mean_3"] - recent_type["level"]
)

RECENT_TYPE_FEATURES = [
    "recent_type_mean_3",
    "recent_type_vs_author",
    "recent_type_std_3"
]

table = table.merge(
    recent_type[["author", "post"] + RECENT_TYPE_FEATURES],
    on=["author", "post"],
    how="left",
    validate="one_to_one"
)

In [ ]:
RECENT_NUMERIC = (NUMERIC + EXTRA_TEXT_FEATURES + ACTIVITY_FEATURES + REACTION_FEATURES + CONTENT_HISTORY_FEATURES + TYPE_PERFORMANCE_FEATURES + RECENT_TYPE_FEATURES)
RECENT_COLUMNS = RECENT_NUMERIC + CATEGORICAL
assert len(RECENT_COLUMNS) == 43
assert len(set(RECENT_COLUMNS)) == len(RECENT_COLUMNS)

def get_tabular(mask):
    X = table.loc[mask, RECENT_COLUMNS].copy()
    X[CATEGORICAL] = X[CATEGORICAL].fillna("unknown")
    return X

X_train_recent = get_tabular(train_mask)
X_val_recent = get_tabular(val_mask)
print("Feature count:", len(RECENT_COLUMNS))
print("Train:", X_train_recent.shape, "Validation:", X_val_recent.shape)

In [ ]:
for forbidden in ["label", "class", "split", "resid", "log_reactions", "post", "author"]:
    assert forbidden not in RECENT_COLUMNS, f"Target or identifier used as feature: {forbidden}"
assert table.loc[full_train_mask, RECENT_COLUMNS].shape[1] == 43
print("Feature leakage guard: OK (direct target and reaction fields excluded)")


## 4. Validation models

CatBoost uses tabular features; logistic regression uses word TF-IDF.


In [ ]:
def make_catboost():
    return CatBoostClassifier(iterations=700, depth=6, learning_rate=0.05, loss_function="MultiClass", class_weights=[1.5, 1.0, 1.5], cat_features=CATEGORICAL, random_seed=42, verbose=False, thread_count=-1)

recent_model = make_catboost()
recent_model.fit(X_train_recent, y_train)
recent_proba = recent_model.predict_proba(X_val_recent)
print("CatBoost validation macro-F1:", f1_score(y_val, recent_model.classes_[recent_proba.argmax(axis=1)], average="macro"))

In [ ]:
vectorizer = TfidfVectorizer(max_features=30000, min_df=5, max_df=0.95, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(table.loc[train_mask, "text"].fillna(""))
X_val_tfidf = vectorizer.transform(table.loc[val_mask, "text"].fillna(""))
text_model = LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs", random_state=42)
text_model.fit(X_train_tfidf, y_train)
text_proba = text_model.predict_proba(X_val_tfidf)
print("TF-IDF validation macro-F1:", f1_score(y_val, text_model.predict(X_val_tfidf), average="macro"))

## 5. Ensemble selection


In [ ]:
validation_results = []
for text_weight in [0.20, 0.30, 0.35, 0.40, 0.50]:
    probabilities = (1 - text_weight) * recent_proba + text_weight * text_proba
    predictions = recent_model.classes_[probabilities.argmax(axis=1)]
    validation_results.append({"text_weight": text_weight, "macro_f1": f1_score(y_val, predictions, average="macro")})
display(pd.DataFrame(validation_results))

In [ ]:
validation_scores = pd.DataFrame(validation_results)
best_weight = validation_scores.loc[validation_scores["macro_f1"].idxmax(), "text_weight"]
assert best_weight in validation_scores["text_weight"].values
assert np.array_equal(recent_model.classes_, text_model.classes_)
print(f"Selected TF-IDF weight: {best_weight:.2f}")


## 6. Extended features and additional models

Add author-relative features (length, engagement trends, timing, and text novelty), character TF-IDF, and CatBoost variants.


In [ ]:
import re
from collections import deque

relative = levelled[["author", "post", "text", "posted_at", "log_reactions", "level", "content_type"]].copy().reset_index(drop=True)
relative["text"] = relative["text"].fillna("").astype(str)
relative["char_log"] = np.log1p(relative["text"].str.len())
relative["word_log"] = np.log1p(relative["text"].str.split().str.len())
relative["has_url"] = relative["text"].str.contains(r"⟨URL⟩|https?://", regex=True).astype(float)
relative["has_question_mark"] = relative["text"].str.contains("?", regex=False).astype(float)
relative["hour_value"] = relative["posted_at"].dt.hour.astype(float)
relative["weekday_value"] = relative["posted_at"].dt.weekday.astype(float)

grouped = relative.groupby("author", sort=False)
for source_column in ["char_log", "word_log", "has_url", "has_question_mark"]:
    previous = grouped[source_column].shift(1)
    relative[f"author_prev_{source_column}"] = previous.groupby(relative["author"]).transform(
        lambda s: s.expanding(min_periods=1).mean()
    )

relative["relative_char_length"] = relative["char_log"] - relative["author_prev_char_log"]
relative["relative_word_length"] = relative["word_log"] - relative["author_prev_word_log"]
relative["unusual_link"] = relative["has_url"] - relative["author_prev_has_url"]
relative["unusual_question"] = relative["has_question_mark"] - relative["author_prev_has_question_mark"]

past_reactions = grouped["log_reactions"].shift(1)
relative["author_mean_reactions_10"] = past_reactions.groupby(relative["author"]).transform(
    lambda s: s.rolling(10, min_periods=1).mean()
)
relative["author_median_reactions_10"] = past_reactions.groupby(relative["author"]).transform(
    lambda s: s.rolling(10, min_periods=1).median()
)
relative["author_std_reactions_10"] = past_reactions.groupby(relative["author"]).transform(
    lambda s: s.rolling(10, min_periods=2).std()
)
relative["author_mean_reactions_20"] = past_reactions.groupby(relative["author"]).transform(
    lambda s: s.rolling(20, min_periods=1).mean()
)
relative["reaction_trend_10_20"] = relative["author_mean_reactions_10"] - relative["author_mean_reactions_20"]
relative["recent_vs_author_level"] = relative["author_mean_reactions_10"] - relative["level"]
relative["recent_median_vs_level"] = relative["author_median_reactions_10"] - relative["level"]

# Performance of an author's earlier posts on the same weekday.
previous_weekday = relative.groupby(["author", "weekday_value"], sort=False)["log_reactions"]
relative["weekday_history_mean"] = previous_weekday.transform(
    lambda s: s.shift(1).expanding(min_periods=1).mean()
)
relative["weekday_history_count"] = relative.groupby(
    ["author", "weekday_value"], sort=False
).cumcount()
relative["weekday_vs_author_level"] = relative["weekday_history_mean"] - relative["level"]

# Historical posting time: circular distance from the author's previous mean hour.
relative["sin_hour"] = np.sin(2 * np.pi * relative["hour_value"] / 24)
relative["cos_hour"] = np.cos(2 * np.pi * relative["hour_value"] / 24)
for part in ["sin_hour", "cos_hour"]:
    previous = grouped[part].shift(1)
    relative[f"previous_{part}"] = previous.groupby(relative["author"]).transform(
        lambda s: s.expanding(min_periods=1).mean()
    )
relative["posting_hour_similarity"] = (
    relative["sin_hour"] * relative["previous_sin_hour"]
    + relative["cos_hour"] * relative["previous_cos_hour"]
)

RELATIVE_FEATURES = [
    "relative_char_length", "relative_word_length", "unusual_link", "unusual_question",
    "author_mean_reactions_10", "author_median_reactions_10", "author_std_reactions_10",
    "author_mean_reactions_20", "reaction_trend_10_20", "recent_vs_author_level",
    "recent_median_vs_level", "weekday_history_mean", "weekday_history_count",
    "weekday_vs_author_level", "posting_hour_similarity"
]

table = table.merge(
    relative[["author", "post"] + RELATIVE_FEATURES],
    on=["author", "post"], how="left", validate="one_to_one"
)
print("Author-relative features:", len(RELATIVE_FEATURES))


In [ ]:
# Lexical novelty with respect to the last five posts of the same author.
# Each post is compared before its own tokens enter the history.
token_pattern = re.compile(r"(?u)\b[^\W\d_][\w]{2,}\b")
novelty = levelled[["author", "post", "text"]].copy().reset_index(drop=True)
novelty["text"] = novelty["text"].fillna("").astype(str)
novelty_scores = np.zeros(len(novelty), dtype=np.float32)
overlap_scores = np.zeros(len(novelty), dtype=np.float32)

for _, author_rows in novelty.groupby("author", sort=False):
    recent_token_sets = deque(maxlen=5)
    for idx, row in author_rows.iterrows():
        tokens = set(token_pattern.findall(row["text"][:2000].lower()))
        if recent_token_sets and tokens:
            previous_tokens = set().union(*recent_token_sets)
            if previous_tokens:
                overlap = len(tokens & previous_tokens)
                overlap_scores[idx] = overlap / len(tokens)
                novelty_scores[idx] = 1 - overlap / len(tokens)
        recent_token_sets.append(tokens)

novelty["lexical_overlap_previous_5"] = overlap_scores
novelty["lexical_novelty_previous_5"] = novelty_scores
LEXICAL_FEATURES = ["lexical_overlap_previous_5", "lexical_novelty_previous_5"]
table = table.merge(
    novelty[["author", "post"] + LEXICAL_FEATURES],
    on=["author", "post"], how="left", validate="one_to_one"
)

EXTENDED_COLUMNS = RECENT_COLUMNS + RELATIVE_FEATURES + LEXICAL_FEATURES
assert len(set(EXTENDED_COLUMNS)) == len(EXTENDED_COLUMNS)
assert not any(c in EXTENDED_COLUMNS for c in ["label", "class", "resid", "log_reactions", "author", "post"])
print("Total tabular features:", len(EXTENDED_COLUMNS))


In [ ]:
# Separate feature accessor for the extended candidate.
def get_extended_tabular(mask):
    X = table.loc[mask, EXTENDED_COLUMNS].copy()
    X[CATEGORICAL] = X[CATEGORICAL].fillna("unknown")
    X[RELATIVE_FEATURES + LEXICAL_FEATURES] = (
        X[RELATIVE_FEATURES + LEXICAL_FEATURES]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(-1.0)
    )
    return X

X_train_extended = get_extended_tabular(train_mask)
X_val_extended = get_extended_tabular(val_mask)
print("Train:", X_train_extended.shape, "Validation:", X_val_extended.shape)


### Character TF-IDF

Character 3–5 grams complement word-based text features.


In [ ]:
char_vectorizer = TfidfVectorizer(
    analyzer="char", ngram_range=(3, 5), max_features=60000,
    min_df=8, max_df=0.98, sublinear_tf=True, dtype=np.float32
)
X_train_char = char_vectorizer.fit_transform(table.loc[train_mask, "text"].fillna(""))
X_val_char = char_vectorizer.transform(table.loc[val_mask, "text"].fillna(""))
char_model = LogisticRegression(
    max_iter=1000, class_weight="balanced", solver="lbfgs", random_state=42
)
char_model.fit(X_train_char, y_train)
char_proba = char_model.predict_proba(X_val_char)
assert np.array_equal(recent_model.classes_, char_model.classes_)
print("Character TF-IDF validation macro-F1:",
      f1_score(y_val, char_model.classes_[char_proba.argmax(axis=1)], average="macro"))


### CatBoost variants

Compare models with depths 5, 6, and 7 and their probability blends.


In [ ]:
CATBOOST_CONFIGS = {
    "extended_depth6": dict(iterations=850, depth=6, learning_rate=0.05,
                            l2_leaf_reg=5, class_weights=[1.5, 1.0, 1.5]),
    "extended_depth7": dict(iterations=900, depth=7, learning_rate=0.045,
                            l2_leaf_reg=7, class_weights=[1.8, 1.0, 1.5]),
    "extended_depth5": dict(iterations=800, depth=5, learning_rate=0.055,
                            l2_leaf_reg=8, class_weights=[1.7, 1.0, 1.6]),
}

def make_extended_catboost(name):
    return CatBoostClassifier(
        loss_function="MultiClass", cat_features=CATEGORICAL,
        random_seed=42, verbose=False, thread_count=-1,
        **CATBOOST_CONFIGS[name]
    )

candidate_models = {"original_43": recent_model}
candidate_probas = {"original_43": recent_proba}

for model_name in CATBOOST_CONFIGS:
    print("Training", model_name, flush=True)
    model = make_extended_catboost(model_name)
    model.fit(X_train_extended, y_train)
    candidate_models[model_name] = model
    candidate_probas[model_name] = model.predict_proba(X_val_extended)
    assert np.array_equal(recent_model.classes_, model.classes_)
    predicted = model.classes_[candidate_probas[model_name].argmax(axis=1)]
    print("Validation CatBoost macro-F1:", f1_score(y_val, predicted, average="macro"))

In [ ]:
from IPython.display import display

# Fixed, interpretable CatBoost blends. The original v2 configuration is
# included unchanged, and selection depends exclusively on validation labels.
CATBOOST_BLENDS = {
    "extended_depth7": {"extended_depth7": 1.0},
    "extended_depth6": {"extended_depth6": 1.0},
    "extended_depth5": {"extended_depth5": 1.0},
    "depth6_depth7": {"extended_depth6": 0.35, "extended_depth7": 0.65},
    "depth5_depth7": {"extended_depth5": 0.35, "extended_depth7": 0.65},
    "depth5_depth6": {"extended_depth5": 0.40, "extended_depth6": 0.60},
    "depth5_depth6_depth7": {
        "extended_depth5": 0.20, "extended_depth6": 0.25,
        "extended_depth7": 0.55
    },
    "original_depth7": {"original_43": 0.25, "extended_depth7": 0.75},
    "original_depth5_depth7": {
        "original_43": 0.20, "extended_depth5": 0.20,
        "extended_depth7": 0.60
    },
}

TEXT_WEIGHTS = [
    (0.30, 0.10), (0.25, 0.10), (0.35, 0.10),
    (0.30, 0.05), (0.30, 0.15),
    (0.35, 0.00), (0.25, 0.15),
]

# Convert a combination of CatBoost models into one validation probability array.
blend_probas = {}
for blend_name, model_weights in CATBOOST_BLENDS.items():
    assert np.isclose(sum(model_weights.values()), 1.0)
    blend_probas[blend_name] = sum(
        weight * candidate_probas[name] for name, weight in model_weights.items()
    )

candidate_rows = []
for blend_name, tabular_proba in blend_probas.items():
    for word_weight, char_weight in TEXT_WEIGHTS:
        tabular_weight = 1 - word_weight - char_weight
        combined = (tabular_weight * tabular_proba
                    + word_weight * text_proba
                    + char_weight * char_proba)
        predicted = recent_model.classes_[combined.argmax(axis=1)]
        candidate_rows.append({
            "blend": blend_name,
            "word_weight": word_weight,
            "char_weight": char_weight,
            "macro_f1": f1_score(y_val, predicted, average="macro"),
            "qwk": cohen_kappa_score(y_val, predicted, weights="quadratic"),
        })

candidate_scores = pd.DataFrame(candidate_rows).sort_values(
    ["macro_f1", "qwk"], ascending=False
).reset_index(drop=True)
candidate_scores.to_csv(RESULTS_DIR / "validation_candidates.csv", index=False)

# Published previous-run configuration: CatBoost depth 7 (60%), word (30%), char (10%).
reference_row = candidate_scores.loc[
    (candidate_scores["blend"] == "extended_depth7")
    & np.isclose(candidate_scores["word_weight"], 0.30)
    & np.isclose(candidate_scores["char_weight"], 0.10)
].iloc[0]
reference_score = float(reference_row["macro_f1"])
best_row = candidate_scores.iloc[0]
minimum_gain = 0.0025

if float(best_row["macro_f1"]) >= reference_score + minimum_gain:
    selected_blend_name = str(best_row["blend"])
    selected_word_weight = float(best_row["word_weight"])
    selected_char_weight = float(best_row["char_weight"])
else:
    selected_blend_name = "extended_depth7"
    selected_word_weight = 0.30
    selected_char_weight = 0.10

selected_catboost_weights = CATBOOST_BLENDS[selected_blend_name]
selected_tabular_weight = 1 - selected_word_weight - selected_char_weight
assert selected_tabular_weight >= 0
print("Reference v2 validation macro-F1:", round(reference_score, 6))
print("Best candidate validation macro-F1:", round(float(best_row["macro_f1"]), 6))
print("Selected CatBoost blend:", selected_blend_name)
print("CatBoost model weights:", selected_catboost_weights)
print("Fusion weights (tabular, word, char):",
      (round(selected_tabular_weight, 2), selected_word_weight, selected_char_weight))
display(candidate_scores.head(15))

### Validation comparison

Keep the original depth-7 ensemble unless a candidate improves validation macro-F1 by at least 0.0025.


In [ ]:
selected_validation_proba = (
    selected_tabular_weight * blend_probas[selected_blend_name]
    + selected_word_weight * text_proba
    + selected_char_weight * char_proba
)
selected_validation_pred = recent_model.classes_[selected_validation_proba.argmax(axis=1)]
reference_validation_proba = (
    0.60 * candidate_probas["extended_depth7"]
    + 0.30 * text_proba
    + 0.10 * char_proba
)
reference_validation_pred = recent_model.classes_[reference_validation_proba.argmax(axis=1)]
reference_validation_f1 = f1_score(y_val, reference_validation_pred, average="macro")
selected_validation_f1 = f1_score(y_val, selected_validation_pred, average="macro")
print("Reference validation macro-F1:", reference_validation_f1)
print("Selected validation macro-F1:", selected_validation_f1)
print(classification_report(y_val, selected_validation_pred, digits=4))

In [ ]:
# Weighted feature importances for the chosen CatBoost blend.
importances = []
for model_name, weight in selected_catboost_weights.items():
    model_features = RECENT_COLUMNS if model_name == "original_43" else EXTENDED_COLUMNS
    values = candidate_models[model_name].get_feature_importance()
    importances.append(pd.DataFrame({
        "feature": model_features,
        "importance": weight * values,
    }))
feature_importance = (pd.concat(importances, ignore_index=True)
                      .groupby("feature", as_index=False)["importance"].sum()
                      .sort_values("importance", ascending=False))
print("Weighted CatBoost feature importance on validation training:")
display(feature_importance.head(15))

## 7. Final training

Retrain the selected models on the full official training split.


In [ ]:
y_train_full = table.loc[full_train_mask, "label"]
final_catboost_models = {}
X_train_recent_full = get_tabular(full_train_mask)
X_test_recent = get_tabular(test_mask)
X_train_extended_full = get_extended_tabular(full_train_mask)
X_test_extended = get_extended_tabular(test_mask)

for model_name in selected_catboost_weights:
    print("Final training:", model_name, flush=True)
    if model_name == "original_43":
        model = make_catboost()
        model.fit(X_train_recent_full, y_train_full)
    else:
        model = make_extended_catboost(model_name)
        model.fit(X_train_extended_full, y_train_full)
    model.save_model(str(RESULTS_DIR / f"final_{model_name}.cbm"))
    final_catboost_models[model_name] = model
    assert np.array_equal(recent_model.classes_, model.classes_)
print("Final CatBoost models:", list(final_catboost_models),
      "| training posts:", len(y_train_full))

In [ ]:
final_text_model = None
final_vectorizer = None
final_char_model = None
final_char_vectorizer = None

if selected_word_weight > 0:
    final_vectorizer = TfidfVectorizer(
        max_features=30000, min_df=5, max_df=0.95,
        ngram_range=(1, 2), sublinear_tf=True
    )
    X_train_tfidf_full = final_vectorizer.fit_transform(table.loc[full_train_mask, "text"].fillna(""))
    X_test_tfidf_full = final_vectorizer.transform(table.loc[test_mask, "text"].fillna(""))
    final_text_model = LogisticRegression(
        max_iter=1000, class_weight="balanced", solver="lbfgs", random_state=42
    )
    final_text_model.fit(X_train_tfidf_full, y_train_full)
    joblib.dump(final_vectorizer, RESULTS_DIR / "final_tfidf_vectorizer.pkl")
    joblib.dump(final_text_model, RESULTS_DIR / "final_tfidf_model.pkl")

if selected_char_weight > 0:
    final_char_vectorizer = TfidfVectorizer(
        analyzer="char", ngram_range=(3, 5), max_features=60000,
        min_df=8, max_df=0.98, sublinear_tf=True, dtype=np.float32
    )
    X_train_char_full = final_char_vectorizer.fit_transform(table.loc[full_train_mask, "text"].fillna(""))
    X_test_char_full = final_char_vectorizer.transform(table.loc[test_mask, "text"].fillna(""))
    final_char_model = LogisticRegression(
        max_iter=1000, class_weight="balanced", solver="lbfgs", random_state=42
    )
    final_char_model.fit(X_train_char_full, y_train_full)
    joblib.dump(final_char_vectorizer, RESULTS_DIR / "final_char_vectorizer.pkl")
    joblib.dump(final_char_model, RESULTS_DIR / "final_char_model.pkl")

print("Full training finished. Text weights:", selected_word_weight, selected_char_weight)


## 8. Test evaluation

Report macro-F1, QWK, accuracy, and per-class scores.


In [ ]:
# Recreate the CatBoost mixture using models retrained on the full train split.
final_tabular_proba = np.zeros((int(test_mask.sum()), 3), dtype=float)
for model_name, weight in selected_catboost_weights.items():
    model = final_catboost_models[model_name]
    X = X_test_recent if model_name == "original_43" else X_test_extended
    final_tabular_proba += weight * model.predict_proba(X)

final_proba = selected_tabular_weight * final_tabular_proba
if selected_word_weight > 0:
    tfidf_proba = final_text_model.predict_proba(X_test_tfidf_full)
    assert np.array_equal(recent_model.classes_, final_text_model.classes_)
    final_proba += selected_word_weight * tfidf_proba
if selected_char_weight > 0:
    char_test_proba = final_char_model.predict_proba(X_test_char_full)
    assert np.array_equal(recent_model.classes_, final_char_model.classes_)
    final_proba += selected_char_weight * char_test_proba

assert np.allclose(final_proba.sum(axis=1), 1.0, atol=1e-5)
final_pred = recent_model.classes_[final_proba.argmax(axis=1)].astype(int)
print("Selected blend:", selected_blend_name)
print("Test macro-F1:", f1_score(y_test, final_pred, average="macro"))
print("Test QWK:", cohen_kappa_score(y_test, final_pred, weights="quadratic"))
print("Test accuracy:", accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred, digits=4))
print(confusion_matrix(y_test, final_pred, labels=[0, 1, 2]))

## 9. Export predictions

Save `author`, `post`, and `label` for all test posts.


In [ ]:
predictions = table.loc[test_mask, ["author", "post"]].copy()
predictions["label"] = final_pred
output_path = RESULTS_DIR / "final_predictions.csv"
predictions.to_csv(output_path, index=False)
assert len(predictions) == int(test_mask.sum())
assert not predictions.duplicated(["author", "post"]).any()
assert predictions["label"].isin([0, 1, 2]).all()
assert set(map(tuple, predictions[["author", "post"]].to_numpy())) == set(map(tuple, labels.loc[labels["split"] == "test", ["author", "post"]].to_numpy()))
assert predictions.columns.tolist() == ["author", "post", "label"]
print("Saved:", output_path)
print("Rows:", len(predictions))
# Official test metric verification (predictions already fixed).
print("Macro-F1 vs official baseline:",
      round(f1_score(y_test, final_pred, average="macro") - 0.5410, 4))


## 10. Results

| Model | Test macro-F1 | Test QWK |
|---|---:|---:|
| Official baseline | 0.5410 | 0.4576 |
| CatBoost + word/character TF-IDF | **0.6319** | **0.5612** |

A three-CatBoost ensemble reached 0.6501 validation macro-F1 but did not meet the preset improvement threshold, so the original depth-7 ensemble was retained.

**Limitations:** No visual media or social graph features; reaction history reflects the crawl-time snapshot.
